# Two-region coupling


## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path


# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import asdict


In [ ]:
from tvb.datatypes.connectivity import Connectivity
from tvb.simulator.coupling import Linear
from tvb.simulator.monitors import Raw


In [ ]:
from aux.environment import resolve_directory
from aux.tvb.stimulus import build_stimulus
from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import DynamicParameterSimulator, build_single_region_simulator
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results


Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Simulation outputs.
OUTPUT_DTYPE = "float32"
OUTPUT_STATES = ["E"]

# El rango del modelo sugiere que las tasas E e I se expresan en kHz.
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L351
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L671
STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²", # Varianzas y covarianza.
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}
# El rango y la suma con Fe sin conversión sugieren kHz para la entrada externa.
# Cálculo de Fe: https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L307
# Suma con external_input: https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L429
# La tabla del artículo y la documentación del modelo indican Hz.
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L116
# https://link.springer.com/article/10.1007/s10827-017-0668-2/tables/1
# En el estímulo compartido, external_input_in_ex replica esta señal y usa la misma unidad.
STIMULUS_UNITS = {"external_input_ex_ex": "kHz"}


In [ ]:
# 1.1) Simulation parameters.
DURATION_MS = 1000.0
DT_MS = 0.5


In [ ]:
# 1.2) Input stimulus: repeated pulses expose both level and timing errors.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse_train": {
            "baseline_khz": 0.005,
            "pulse_amplitude_khz": 0.04,
            "pulse_onset_ms": 200.0,
            "pulse_duration_ms": 160.0,
            "pulse_period_ms": 320.0,
        },
    },
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"

# Referencias para la definición de parámetros y sus valores por defecto:
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L60
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L148
# https://link.springer.com/article/10.1007/s10827-017-0668-2/tables/1

MODEL_PARAMETERS = {
    # Población
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Celulares: propiedades eléctricas
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Celulares: parámetros de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 0.0, 
    "tau_w_i": 1.0, #!No puede ser 0 como en la tabla

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    # Celulares: escala de tiempo
    "T": 10.0,

    # Parámetro del modelo de segundo orden; las simulaciones de este cuaderno usan primer orden.
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Coeficientes de la función de transferencia: phenomenological threshold coefficients
    "P_e": [-0.04983106, 0.005063550882777035, -0.023470121807314552, 0.0022951513725067503, -0.0004105302652029825, 0.010547051343547399, -0.03659252821136933, 0.007437487505797858, 0.001265064721846073, -0.04072161294490446],
    "P_i": [-0.05149122024209484, 0.004003689190271077, -0.008352013668528155, 0.0002414237992765705, -0.0005070645080016026, 0.0014345394104282397, -0.014686689498949967, 0.004502706285435741, 0.0028472190352532454, -0.015357804594594548],

    # Parámetros de ruido: ausentes de la tabla citada. Este cuaderno usa integración determinista.
    "weight_noise": 2e-4, #10.5
    "tau_OU": 2, #0.5
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


## Build the stimulus


In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)

input_signal = stimulus[0][1][:, 0]


## Deterministic Simulation


### Compare model parameters


In [ ]:
# These values include the 01b defaults and a focused search around lower external gain.
PARAMETER_CONFIGS = {
    # "Defaults": {},
    # "Lower K_ext_e": {"K_ext_e": 100}, # reduce Khz
    # "Lower T": {"K_ext_e": 100, "T": 1.2},   # trasiciona más rápido. Si lo bajas más tiene más ruido
    "lower Q_e": {"K_ext_e": 100, "T": 1.2, "Q_e": 0.5}, # reduce Khz
    "last": {"K_ext_e": 120, "T": 1.2, "Q_e": 0.5}, # reduce Khz
}



In [ ]:
integrator = build_integrator(DT_MS, integrator_class="HeunDeterministic")
results = {}
for name, changes in PARAMETER_CONFIGS.items():
    parameters = MODEL_PARAMETERS | changes
    model = build_model(parameters, INTEGER_PARAMETERS,
                        model_class="ZerlautAdaptationFirstOrder",
                        variable_of_interest=OUTPUT_STATES)
    
    simulator = build_single_region_simulator(model, stimulus, integrator)
    time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
    results[name] = raw[:, :, 0, 0]
    pprint(f"{name}: {len(time_sim)} samples")


### Plot results


In [ ]:
metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(name="ZerlautAdaptationFirstOrder", parameters=MODEL_PARAMETERS,
                        integer_parameters=list(INTEGER_PARAMETERS),
                        output_variables=OUTPUT_STATES, state_units=STATE_UNITS,
                        stimulus_units=STIMULUS_UNITS),
    stimulus=STIMULUS,
)
fig = plt.figure(figsize=(4.5 * len(results), 6), constrained_layout=True)
grid = fig.add_gridspec(1, len(results))
for column, (name, output_E) in enumerate(results.items()):
    _, axes = plot_results(time_sim, {"E": output_E}, metadata,
                           show_stimulus=True, ax=fig.add_subplot(grid[0, column]),
                           plot_states=["E"])
    axes[0].set_title(name)
plt.show()


## Two-region Simulation


<font color=ff0000>No lo entiendo</font> <font color=00ff00>No entendia</font>

$$
\begin{aligned}
f_e^{(r)} =& (\nu_e+10^{-6})(1-g)p_eN_{\mathrm{tot}} + \nu_e^{\mathrm{ext},(r)}K_e^{\mathrm{ext}},\\
f_i^{(r)} =& (\nu_i+10^{-6})gp_iN_{\mathrm{tot}}+\nu_i^{\mathrm{ext},(r)}K_i^{\mathrm{ext}},
\qquad r\in\{e,i\}.
\end{aligned}
$$


$$
\begin{aligned}
\nu_e^{\mathrm{ext},(r)} =& \boxed{c_0} + k_{\mathrm{local}}\nu_e + w_{\mathrm{noise}}x_{\mathrm{OU}} + \boxed{\nu^{\mathrm{drive}}_{e\to(r)}}\\
\nu_i^{\mathrm{ext},(r)} =& k_{\mathrm{local}}\nu_i + \boxed{\nu^{\mathrm{drive}}_{i\to(r)}}.
\end{aligned}
$$

In [ ]:
def build_two_region_simulator(input_parameters,output_parameters,
                               stimulus, integrator,
                               model_class="ZerlautAdaptationFirstOrder",
                               output_states=OUTPUT_STATES):

    #ToDo: La mayoría de estos parametros se sobrescrbien luego
    model = build_model(output_parameters, INTEGER_PARAMETERS,
                        model_class=model_class,
                        variable_of_interest=output_states)

    # ToDo: Se esta sobrescribiendo lo anterioro
    for name, value in input_parameters.items():
        dtype = np.int64 if name in INTEGER_PARAMETERS else np.float64
        if np.ndim(value):
            # (valores, regiones, modos)
            values = np.stack([value, output_parameters[name]], axis=1)[:, :, np.newaxis]
        else:
            values = np.array([value, output_parameters[name]], dtype=dtype)
        setattr(model, name, np.asarray(values, dtype=dtype))
        

    dynamic_parameters = [
        (name, np.column_stack([signal[:, 0], np.zeros_like(signal[:, 0])]))
        for name, signal in stimulus
    ]
    
    connectivity = Connectivity(
        weights=np.array([[0.0, 0.0], [1.0, 0.0]]),
        tract_lengths=np.zeros((2, 2)), # distancia 0, delay 0
        speed=np.array([3.0]),
        centres=np.zeros((2, 3)),
        region_labels=np.array(["input", "output"]),
    )
    return DynamicParameterSimulator(
        model=model, dynamic_parameters=dynamic_parameters,
        connectivity=connectivity, coupling=Linear(a=np.array([1.0])),
        integrator=integrator, monitors=(Raw(),),
        initial_conditions=np.zeros((1, len(model.state_variables), 2, 1)),
    ).configure()


In [ ]:

two_region_simulator = build_two_region_simulator(
    MODEL_PARAMETERS | PARAMETER_CONFIGS["last"] , MODEL_PARAMETERS,
    stimulus, integrator
)
time_two, raw_two = two_region_simulator.run(n_steps=len(time_ms))[0]
input_E = raw_two[:, :, 0, 0]
output_E = raw_two[:, :, 1, 0]

pprint("Connection weights:", two_region_simulator.connectivity.weights)
pprint("Connection delays (ms):", two_region_simulator.connectivity.delays)


### Plot results


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
axes[0].stairs(input_signal, np.r_[time_ms, DURATION_MS], label="external_input_ex_ex")
axes[1].plot(time_two, input_E, label="Input E")
axes[2].plot(time_two, output_E, label="Output E")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
    axis.set_ylabel("kHz")
axes[-1].set_xlabel("Time (ms)")
plt.show()


## Excitatory-inhibitory stimulus


In [ ]:
SHARED_STIMULUS = STIMULUS | {
    "external_input_in_ex": {"external_input_ex_ex": 1.0},
}

time_shared, shared_stimulus = build_stimulus(SHARED_STIMULUS, DURATION_MS, DT_MS)

shared_model = build_model(MODEL_PARAMETERS, INTEGER_PARAMETERS,
                           model_class="ZerlautAdaptationFirstOrder",
                           variable_of_interest=OUTPUT_STATES)
shared_simulator = build_single_region_simulator(shared_model, shared_stimulus, integrator)
time_shared_sim, raw_shared = shared_simulator.run(n_steps=len(time_shared))[0]
raw_E = raw_shared[:, :, 0, 0]



### Plot results

In [ ]:
shared_metadata = ResultMetadata(
    simulation=metadata.simulation,
    model=ModelMetadata(name="ZerlautAdaptationFirstOrder", parameters=MODEL_PARAMETERS,
                        integer_parameters=list(INTEGER_PARAMETERS),
                        output_variables=OUTPUT_STATES, state_units=STATE_UNITS,
                        stimulus_units=STIMULUS_UNITS | {"external_input_in_ex": "kHz"}),
    stimulus=SHARED_STIMULUS,
)
plot_results(time_shared_sim, {"E": raw_E}, shared_metadata)
plt.show()
